# CS781 Capstone · Phase 7 — Full fine-tuning

**Task:** Prior Case Retrieval (PCR), using the original, personally trained **Phase-1 Model 2**.

**Experiment:** `full` — Train all Phase-1 backbone weights and the new retrieval projection head.

### Official evaluation

- **Primary:** Micro-F1@7 (maximize the number of true relevant cases in the top seven).
- **Secondary:** Micro-Precision@7 and Micro-Recall@7.
- Output one `prediction.json` mapping each test query ID to exactly **seven unique test-pool candidate IDs**, in ranked order.

**Data isolation:** Train positives/negatives come only from `Train/`; checkpoint and lexical/neural mixture selection use only `Dev/`; unlabeled `Test/` is prediction-only. No third-party legal dataset or pretrained LM is used.

> This notebook contains the executable Phase-1 Model 2 architecture/ByteLevelBPE loader so the notebook is **standalone** on Kaggle/Colab. You must attach your own Phase-1 checkpoint and tokenizer. Do not run with random Phase-1 weights.


## 1. Environment and dependency check

On Kaggle: enable a GPU, turn on Internet, and add the Hugging Face dataset access token under **Secrets → HF**. Enable access to that secret. If your official dataset is already downloaded, set `DATA_DIR` in the next cell instead.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "huggingface_hub": "huggingface_hub>=0.25",
    "sklearn": "scikit-learn>=1.2",
    "tqdm": "tqdm>=4.60",
}

missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages]
    )

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
CUDA available: True


## 2. Experiment configuration

Set `ASSET_DIR` to the Kaggle dataset containing your **original Model 2** `pytorch_model.bin` and `custom_bpe_scratch.json`. Change `DATA_DIR` only if you already have the official Phase-7 JSON files locally. The default uses the authenticated official Hugging Face repository.

**For KD:** first run the full fine-tuning notebook, publish/attach its `best_model.pt`, and update `TEACHER_CHECKPOINT` below. A random teacher is explicitly rejected.


In [2]:
EXPERIMENT = "full"

# Point to the exact Kaggle input folder with your original Phase-1 Model 2 files.
# An unambiguous search under /kaggle/input is permitted as the default.
ASSET_DIR = "/kaggle/input"

# Leave empty to download the official Phase-7 JSON files using Kaggle Secret HF.
DATA_DIR = ""

# Only required by the KD experiment: trained full-retrieval best_model.pt.
TEACHER_CHECKPOINT = ""

OUTPUT_DIR = "/kaggle/working/phase7_pcr_full"

# Change only if you have a documented, checkpoint-compatible architecture.
# These defaults match the user's original Model 2 checkpoint.
MODEL_VOCAB_SIZE = 32000
MODEL_HIDDEN_DIM = 512
MODEL_LAYERS = 15
MODEL_HEADS = 8
MODEL_KV_HEADS = 4

# Conservative Kaggle GPU settings; adjust only using development evidence.
TRAIN_EPOCHS = 3
MAX_TRAIN_QUERIES = 500
STEPS_PER_EPOCH = 130
TRAIN_BATCH_SIZE = 2
ACCUMULATE_STEPS = 8

# Top-N lexical candidates are reranked using Phase-1 legal embeddings.
RERANK_TOP_N = 60
DEV_QUERIES_PER_EPOCH = 48
CHUNKS_PER_CASE = 2
TOKENS_PER_CHUNK = 160

## 3. Original Phase-1 Model 2 tokenizer and architecture

This is the compatible custom BPE, RoPE, RMSNorm, grouped-query attention (GQA), QK normalization and SiLU-FFN implementation. Loading uses `strict=True`; a missing or mismatched Phase-1 checkpoint is a hard error rather than silently training a different model.


In [3]:
"""Phase-1 Model 2 compatibility layer (RoPE + RMSNorm + GQA + QK-Norm).

Architecture and state_dict names match the user's earlier Phase-1 Model 2
implementation. No pretrained third-party weights are used.
"""

from __future__ import annotations

import json
import math
import re
from collections import Counter
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


class CustomByteLevelBPE:
    """Loader/encoder for the user's original custom byte-level BPE JSON."""

    def __init__(self, vocab_size: int = 32000):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.pad_id = 0

    def load(self, path: str | Path) -> None:
        with open(path, "r", encoding="utf-8") as handle:
            data = json.load(handle)

        self.vocab_size = int(data["vocab_size"])
        self.merges = {
            tuple(map(int, pair.split(","))): int(new_id)
            for pair, new_id in data["merges"].items()
        }
        self.vocab = {
            int(token_id): bytes(byte_values)
            for token_id, byte_values in data["vocab"].items()
        }

    @staticmethod
    def _merge_vocab(ids: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
        result = []
        index = 0

        while index < len(ids):
            if index + 1 < len(ids) and (ids[index], ids[index + 1]) == pair:
                result.append(new_id)
                index += 2
            else:
                result.append(ids[index])
                index += 1

        return result

    def encode(self, text: str) -> list[int]:
        encoded = []

        for segment in re.findall(r"\S+|\s+", text):
            ids = list(segment.encode("utf-8"))

            while len(ids) >= 2:
                present_pairs = set(zip(ids, ids[1:]))
                pair = min(
                    present_pairs,
                    key=lambda candidate: self.merges.get(candidate, math.inf),
                )

                if pair not in self.merges:
                    break

                ids = self._merge_vocab(ids, pair, self.merges[pair])

            encoded.extend(ids)

        return encoded

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.square().mean(-1, keepdim=True) + self.eps) * self.weight


def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0) -> torch.Tensor:
    freqs = 1.0 / theta ** (torch.arange(0, dim, 2).float() / dim)
    times = torch.arange(end)
    phases = torch.outer(times, freqs)
    return torch.view_as_real(torch.polar(torch.ones_like(phases), phases))


def apply_rotary_emb(
    queries: torch.Tensor,
    keys: torch.Tensor,
    freqs: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    queries_complex = torch.view_as_complex(
        queries.float().reshape(*queries.shape[:-1], -1, 2)
    )
    keys_complex = torch.view_as_complex(
        keys.float().reshape(*keys.shape[:-1], -1, 2)
    )
    freqs = freqs.view(1, queries_complex.shape[1], 1, queries_complex.shape[-1])

    q_rotated = torch.view_as_real(queries_complex * freqs).flatten(3).to(queries.dtype)
    k_rotated = torch.view_as_real(keys_complex * freqs).flatten(3).to(keys.dtype)

    return q_rotated, k_rotated

In [5]:
class ManualTransformerAttention(nn.Module):
    def __init__(self, head_dim: int):
        super().__init__()
        self.scale = 1.0 / math.sqrt(head_dim)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        is_causal: bool = True,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        scores = q @ k.transpose(-2, -1).contiguous()
        scores = scores * self.scale

        if is_causal and q.shape[-2] > 1:
            causal_mask = torch.tril(
                torch.ones(q.shape[-2], k.shape[-2], device=q.device, dtype=torch.bool)
            )
            scores = scores.masked_fill(~causal_mask, -torch.inf)

        if attention_mask is not None:
            key_mask = attention_mask[:, None, None, :].to(torch.bool)
            scores = scores.masked_fill(~key_mask, -torch.inf)

        weights = F.softmax(scores.float(), dim=-1).to(q.dtype)
        return weights @ v.contiguous()

In [6]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, dim: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads

        self.wq = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)
        self.q_norm = RMSNorm(n_heads * self.head_dim)
        self.k_norm = RMSNorm(n_kv_heads * self.head_dim)
        self.manual_attention = ManualTransformerAttention(self.head_dim)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        kv_cache=None,
        attention_mask: torch.Tensor | None = None,
    ):
        batch, length, _ = x.shape
        q = self.q_norm(self.wq(x)).view(
            batch, length, self.n_heads, self.head_dim
        )
        k = self.k_norm(self.wk(x)).view(
            batch, length, self.n_kv_heads, self.head_dim
        )
        v = self.wv(x).view(batch, length, self.n_kv_heads, self.head_dim)
        q, k = apply_rotary_emb(q, k, freqs_cis)

        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat((past_k, k), dim=1)
            v = torch.cat((past_v, v), dim=1)

        new_cache = (k, v)
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=2)

        attended = self.manual_attention(
            q=q.transpose(1, 2),
            k=k.transpose(1, 2),
            v=v.transpose(1, 2),
            is_causal=length > 1,
            attention_mask=attention_mask,
        )
        attended = attended.transpose(1, 2).contiguous().view(batch, length, -1)
        return self.wo(attended), new_cache

In [7]:
class CustomTransformerBlock(nn.Module):
    def __init__(self, dim: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)
        self.attn = GroupedQueryAttention(dim, n_heads, n_kv_heads)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4, bias=False),
            nn.SiLU(),
            nn.Linear(dim * 4, dim, bias=False),
        )

    def forward(self, x, freqs_cis, kv_cache=None, attention_mask=None):
        attn_out, next_cache = self.attn(
            self.norm1(x),
            freqs_cis,
            kv_cache=kv_cache,
            attention_mask=attention_mask,
        )
        hidden = x + attn_out
        return hidden + self.ffn(self.norm2(hidden)), next_cache

In [8]:
class CustomLegalModel(nn.Module):
    """Strictly compatible state_dict with the user's Model 2 Phase-1 model."""

    def __init__(
        self,
        vocab_size: int = 32000,
        dim: int = 512,
        n_layers: int = 15,
        n_heads: int = 8,
        n_kv_heads: int = 4,
        max_seq_len: int = 2048,
    ):
        super().__init__()
        self.tok_embeddings = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList(
            CustomTransformerBlock(dim, n_heads, n_kv_heads)
            for _ in range(n_layers)
        )
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim, vocab_size, bias=False)
        self.output.weight = self.tok_embeddings.weight
        self.register_buffer("freqs_cis", precompute_freqs_cis(dim // n_heads, max_seq_len))
        self.gradient_checkpointing = False

    def forward_hidden(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        length = input_ids.shape[1]
        freqs = torch.view_as_complex(self.freqs_cis[:length].to(input_ids.device))
        hidden = self.tok_embeddings(input_ids)

        for layer in self.layers:
            if self.gradient_checkpointing and self.training and torch.is_grad_enabled():
                def layer_fn(x, selected_layer=layer):
                    return selected_layer(
                        x,
                        freqs,
                        kv_cache=None,
                        attention_mask=attention_mask,
                    )[0]

                hidden = checkpoint(layer_fn, hidden, use_reentrant=False)
            else:
                hidden, _ = layer(
                    hidden,
                    freqs,
                    kv_cache=None,
                    attention_mask=attention_mask,
                )

        return self.norm(hidden)

    def forward(self, tokens, start_pos=0, kv_caches=None, attention_mask=None):
        if start_pos != 0 or kv_caches is not None:
            raise NotImplementedError("Retrieval adapter does not implement incremental caching")
        hidden = self.forward_hidden(tokens, attention_mask=attention_mask)
        return self.output(hidden), []


def load_phase1_model(path: str | Path, **model_kwargs) -> CustomLegalModel:
    """Reject incompatible and partial Phase-1 checkpoints."""
    model = CustomLegalModel(**model_kwargs)
    checkpoint_data = torch.load(path, map_location="cpu", weights_only=True)

    if isinstance(checkpoint_data, dict):
        for key in ("model_state_dict", "state_dict"):
            if key in checkpoint_data:
                checkpoint_data = checkpoint_data[key]
                break

    if not isinstance(checkpoint_data, dict):
        raise TypeError("Phase-1 checkpoint must contain a PyTorch state_dict")

    cleaned = {
        name.removeprefix("module."): value
        for name, value in checkpoint_data.items()
        if isinstance(value, torch.Tensor)
    }
    model.load_state_dict(cleaned, strict=True)
    return model

## 4. Retrieval pipeline

The following cells provide split-safe data handling, legal-text chunking, a lexical reference ranker, a trainable Phase-1 dual encoder, hard-negative contrastive learning, development-only weight selection, and strict Codabench output checks. Every code section is executable in order; no separate `.py` upload is needed.


### 4.1 Core imports and experiment configuration

In [9]:
"""Phase 7 prior-case retrieval: shared, self-contained experiment implementation.

All training examples originate from the official Phase_7_PCR dataset.
The legal encoder is the student's own Phase-1 Model 2, NOT a third-party pretrained LM.
"""

from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import re
import time
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

### 4.2 Reproducibility

In [10]:
# ------------------------------ Reproducibility ----------------------------- #


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class Config:
    experiment: str = "lorma"  # full, partial, lorma, ia3, kd
    seed: int = 42
    repo_id: str = "Exploration-Lab/CS-781-Capstone"
    checkpoint_name: str = "pytorch_model.bin"
    tokenizer_name: str = "custom_bpe_scratch.json"
    asset_dir: str = "/kaggle/input"
    work_dir: str = "/kaggle/working/phase7_pcr"
    data_dir: str = ""  # optional local Phase_7_PCR folder
    teacher_checkpoint: str = ""  # required for KD; trained Phase-7 full model

    vocab_size: int = 32000
    hidden_dim: int = 512
    n_layers: int = 15
    n_heads: int = 8
    n_kv_heads: int = 4
    model_context: int = 2048
    project_dim: int = 256

    chunks_per_doc: int = 2
    tokens_per_chunk: int = 160
    lexical_chars: int = 9000
    rerank_top_n: int = 60
    dev_queries_per_epoch: int = 48
    max_train_queries: int = 500
    epochs: int = 3
    max_steps_per_epoch: int = 130
    batch_size: int = 2
    accumulate_steps: int = 8
    encode_batch_size: int = 8
    learning_rate: float = 0.00025
    backbone_learning_rate: float = 0.000016
    weight_decay: float = 0.01
    patience: int = 2
    temperature: float = 0.08
    kd_temperature: float = 2.0
    kd_weight: float = 0.6
    use_amp: bool = True
    gradient_checkpointing: bool = True
    lorma_rank: int = 8
    lorma_alpha: float = 16.0
    student_layers: int = 8
    top_k: int = 7
    make_test_predictions: bool = True

    def __post_init__(self) -> None:
        if self.experiment not in {"full", "partial", "lorma", "ia3", "kd"}:
            raise ValueError(f"Unsupported experiment: {self.experiment}")
        if self.top_k != 7:
            raise ValueError("Official Phase 7 submissions require exactly seven candidates")
        if self.rerank_top_n < self.top_k:
            raise ValueError("rerank_top_n must be at least 7")
        if self.chunks_per_doc < 1 or self.tokens_per_chunk < 8:
            raise ValueError("Invalid document chunk configuration")

### 4.3 Official data access and JSON parsing

In [11]:
# ---------------------------- Data access / parsing ------------------------- #


FILES = {
    "train_queries": "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev_queries": "Phase_7_PCR/Dev/dev.json",
    "dev_candidates": "Phase_7_PCR/Dev/dev_candidates.json",
    "test_queries": "Phase_7_PCR/Test/test.json",
    "test_candidates": "Phase_7_PCR/Test/test_candidates.json",
}


def get_huggingface_token() -> str | None:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token

    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None


def resolve_data_file(relative_path: str, config: Config) -> Path:
    """Prefer explicitly mounted official files, then download using HF token."""
    filename = Path(relative_path).name

    if config.data_dir:
        root = Path(config.data_dir)
        for candidate in (root / relative_path, root / Path(relative_path).relative_to("Phase_7_PCR")):
            if candidate.is_file():
                return candidate

    target = Path(config.work_dir) / "official_data" / relative_path
    if target.is_file():
        return target

    try:
        from huggingface_hub import hf_hub_download

        downloaded = hf_hub_download(
            repo_id=config.repo_id,
            repo_type="dataset",
            filename=relative_path,
            token=get_huggingface_token(),
        )
        return Path(downloaded)
    except Exception as error:
        raise FileNotFoundError(
            f"Unable to access official file {relative_path!r}. "
            "Set Kaggle Secret HF (read permission), enable internet, or set DATA_DIR "
            "to your downloaded official Phase_7_PCR directory. "
            f"Original error: {error}"
        ) from error


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def unpack_records(payload: Any, *, kind: str) -> dict[str, dict[str, Any]]:
    """Keep ID strings and their leading zeros; handle list- and map-form JSON."""
    if isinstance(payload, dict):
        for collection_key in (kind, "queries", "candidates", "data", "records", "cases"):
            if collection_key in payload and isinstance(payload[collection_key], (list, dict)):
                return unpack_records(payload[collection_key], kind=kind)

        if any(key in payload for key in ("id", "case_id", "query_id", "candidate_id")):
            record_id = next(
                str(payload[key])
                for key in ("id", "case_id", "query_id", "candidate_id")
                if key in payload
            )
            return {record_id: dict(payload)}

        return {
            str(record_id): dict(record) if isinstance(record, dict) else {"text": record}
            for record_id, record in payload.items()
        }

    if not isinstance(payload, list):
        raise TypeError(f"Unexpected {kind} structure: {type(payload).__name__}")

    records = {}
    for record in payload:
        if not isinstance(record, dict):
            raise TypeError(f"Each {kind} record must be an object; got {type(record).__name__}")

        possible_ids = ("id", "case_id", "query_id", "candidate_id", "doc_id")
        record_id = next((record[key] for key in possible_ids if key in record), None)
        if record_id is None:
            raise ValueError(f"A {kind} record is missing an ID: {str(record)[:200]}")
        if str(record_id) in records:
            raise ValueError(f"Duplicate {kind} ID: {record_id}")

        records[str(record_id)] = record

    return records


def extract_sentences(value: Any) -> list[str]:
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        result = []
        for item in value:
            result.extend(extract_sentences(item))
        return result
    if isinstance(value, dict):
        for key in ("sentences", "text", "document", "content", "sentence"):
            if key in value:
                return extract_sentences(value[key])
        return []
    return []


def document_sentences(record: dict[str, Any]) -> list[str]:
    for key in ("text", "sentences", "document", "content", "case_text"):
        if key in record:
            return [sentence.strip() for sentence in extract_sentences(record[key]) if sentence.strip()]
    return []


def document_text(record: dict[str, Any]) -> str:
    return " ".join(document_sentences(record))


def relevant_case_ids(record: dict[str, Any]) -> list[str]:
    value = record.get("relevant_candidates", record.get("relevant_candidate_ids", []))
    if value is None:
        return []
    if isinstance(value, dict):
        value = [key for key, is_relevant in value.items() if is_relevant]
    if not isinstance(value, list):
        raise TypeError("relevant_candidates should be a list or mapping")

    result = []
    for item in value:
        if isinstance(item, dict):
            item = next(
                (item[key] for key in ("id", "candidate_id", "case_id") if key in item),
                None,
            )
        if item is not None:
            result.append(str(item))
    return list(dict.fromkeys(result))


def load_official_data(config: Config) -> dict[str, dict[str, dict[str, Any]]]:
    result = {}
    for name, relpath in FILES.items():
        payload = load_json(resolve_data_file(relpath, config))
        result[name] = unpack_records(
            payload,
            kind="queries" if "queries" in name else "candidates",
        )

    for split in ("train", "dev", "test"):
        queries = result[f"{split}_queries"]
        candidates = result[f"{split}_candidates"]
        if len(candidates) < 7:
            raise ValueError(f"{split} candidate pool contains fewer than 7 cases")
        if not queries:
            raise ValueError(f"{split} query set is empty")
        if not any(document_text(record) for record in queries.values()):
            raise ValueError(f"No usable legal text found for {split} queries")
        if not any(document_text(record) for record in candidates.values()):
            raise ValueError(f"No usable legal text found for {split} candidates")

        print(f"{split.title():5}: {len(queries):5d} queries; {len(candidates):6d} candidates")

    if not any(relevant_case_ids(row) for row in result["train_queries"].values()):
        raise ValueError("Training queries have no relevant_candidates labels")

    if not any(relevant_case_ids(row) for row in result["dev_queries"].values()):
        raise ValueError("Development queries have no relevant_candidates labels")

    return result

### 4.4 Distinct adaptation methods

In [12]:
# ------------------------------- Model adaptation --------------------------- #


class LoRMAPlusLinear(nn.Module):
    """LoRMA+: (I + alpha/r * B A) W0 x, frozen W0, identity at init.

    This is multiplicative adaptation, not additive LoRA applied to x.
    """

    def __init__(self, original: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.base = original
        self.base.requires_grad_(False)
        self.rank = rank
        self.scaling = alpha / rank

        self.a = nn.Linear(original.out_features, rank, bias=False)
        self.b = nn.Linear(rank, original.out_features, bias=False)
        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))
        nn.init.zeros_(self.b.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        frozen_output = self.base(x)
        return frozen_output + self.scaling * self.b(self.a(frozen_output))


class IA3Linear(nn.Module):
    """IA³-style learned channel-wise multiplicative activation rescaling."""

    def __init__(self, original: nn.Linear):
        super().__init__()
        self.base = original
        self.base.requires_grad_(False)
        self.scales = nn.Parameter(torch.ones(original.out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) * self.scales


class RetrievalEncoder(nn.Module):
    def __init__(self, backbone: CustomLegalModel, project_dim: int):
        super().__init__()
        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.LayerNorm(backbone.tok_embeddings.embedding_dim),
            nn.Linear(backbone.tok_embeddings.embedding_dim, project_dim),
        )

    def forward(self, tokens: torch.Tensor, mask: torch.Tensor, doc_indices: torch.Tensor, n_docs: int):
        hidden = self.backbone.forward_hidden(tokens, attention_mask=mask)
        token_weights = mask.unsqueeze(-1).to(hidden.dtype)

        mean_pool = (hidden * token_weights).sum(dim=1)
        mean_pool = mean_pool / token_weights.sum(dim=1).clamp_min(1)
        last_positions = mask.long().sum(dim=1).sub(1).clamp_min(0)
        last_pool = hidden[torch.arange(len(tokens), device=tokens.device), last_positions]

        chunk_embeddings = self.projection(0.5 * mean_pool + 0.5 * last_pool)
        doc_embeddings = torch.zeros(
            (n_docs, chunk_embeddings.shape[-1]),
            device=chunk_embeddings.device,
            dtype=chunk_embeddings.dtype,
        )
        doc_counts = torch.zeros((n_docs, 1), device=tokens.device, dtype=chunk_embeddings.dtype)

        doc_embeddings.index_add_(0, doc_indices, chunk_embeddings)
        doc_counts.index_add_(0, doc_indices, torch.ones_like(doc_indices[:, None], dtype=chunk_embeddings.dtype))

        doc_embeddings = doc_embeddings / doc_counts.clamp_min(1)
        return F.normalize(doc_embeddings.float(), p=2, dim=-1)


def make_model(config: Config, checkpoint: Path, *, method: str | None = None) -> RetrievalEncoder:
    method = method or config.experiment
    backbone = load_phase1_model(
        checkpoint,
        vocab_size=config.vocab_size,
        dim=config.hidden_dim,
        n_layers=config.n_layers,
        n_heads=config.n_heads,
        n_kv_heads=config.n_kv_heads,
        max_seq_len=config.model_context,
    )
    backbone.gradient_checkpointing = config.gradient_checkpointing

    if method == "kd_student":
        indices = np.linspace(0, config.n_layers - 1, config.student_layers).round().astype(int)
        backbone.layers = nn.ModuleList([backbone.layers[int(index)] for index in indices])

    for parameter in backbone.parameters():
        parameter.requires_grad = method == "full"

    if method == "partial":
        for layer in backbone.layers[-4:]:
            layer.requires_grad_(True)
        backbone.norm.requires_grad_(True)

    if method in {"lorma", "ia3"}:
        for layer in backbone.layers:
            attention = layer.attn
            if method == "lorma":
                attention.wq = LoRMAPlusLinear(attention.wq, config.lorma_rank, config.lorma_alpha)
                attention.wv = LoRMAPlusLinear(attention.wv, config.lorma_rank, config.lorma_alpha)
                attention.wo = LoRMAPlusLinear(attention.wo, config.lorma_rank, config.lorma_alpha)
                layer.ffn[0] = LoRMAPlusLinear(
                    layer.ffn[0], config.lorma_rank, config.lorma_alpha
                )
            else:
                attention.wv = IA3Linear(attention.wv)
                attention.wo = IA3Linear(attention.wo)
                layer.ffn[2] = IA3Linear(layer.ffn[2])

    return RetrievalEncoder(backbone, config.project_dim)

### 4.5 Long-case tokenization and document representations

In [13]:
# ----------------------------- Document representations --------------------- #


class DocumentTokenizer:
    def __init__(self, tokenizer: CustomByteLevelBPE, config: Config):
        self.tokenizer = tokenizer
        self.config = config
        self.cache = {}
        self.cache_size = 9000

    def _selected_windows(self, sentences: list[str]) -> list[str]:
        if not sentences:
            return ["[EMPTY DOCUMENT]"]

        windows = [" ".join(sentences[:12])]
        if self.config.chunks_per_doc > 1:
            citation_pattern = re.compile(
                r"\b(?:scc|air|section|sections|article|statute|judgment|\bv\.\b|"
                r"appeal|court|tribunal|held|precedent|versus)\b",
                flags=re.IGNORECASE,
            )
            salient = [
                sentence
                for sentence in sentences[12:]
                if citation_pattern.search(sentence)
            ]
            salient_window = " ".join(salient[:5]) if salient else " ".join(sentences[-12:])
            windows.append(salient_window)

        if self.config.chunks_per_doc > 2:
            windows.append(" ".join(sentences[-12:]))

        return windows[: self.config.chunks_per_doc]

    def __call__(self, record: dict[str, Any]) -> list[list[int]]:
        sentences = document_sentences(record)
        windows = self._selected_windows(sentences)
        cache_key = hashlib.blake2b("\n".join(windows).encode("utf-8"), digest_size=16).hexdigest()

        if cache_key in self.cache:
            return self.cache[cache_key]

        all_chunks = []
        for window in windows:
            ids = self.tokenizer.encode(window[:5000])[: self.config.tokens_per_chunk]
            all_chunks.append(ids or [32])  # space byte; 0 is reserved as a pad token

        if len(self.cache) >= self.cache_size:
            self.cache.clear()
        self.cache[cache_key] = all_chunks
        return all_chunks


def tensorize_docs(records: list[dict[str, Any]], encoder: DocumentTokenizer, device):
    chunks, doc_indices = [], []
    for index, record in enumerate(records):
        for chunk in encoder(record):
            chunks.append(chunk)
            doc_indices.append(index)

    maximum = max(len(chunk) for chunk in chunks)
    tokens = torch.zeros((len(chunks), maximum), dtype=torch.long)
    mask = torch.zeros_like(tokens)

    for index, chunk in enumerate(chunks):
        tokens[index, : len(chunk)] = torch.tensor(chunk, dtype=torch.long)
        mask[index, : len(chunk)] = 1

    return (
        tokens.to(device, non_blocking=True),
        mask.to(device, non_blocking=True),
        torch.tensor(doc_indices, dtype=torch.long, device=device),
    )


def embed_documents(model, documents, tokenizer, device, batch_size, *, show_progress=False):
    """Encode a sequence of records, preserving their order."""
    model.eval()
    vectors = []
    indices = range(0, len(documents), batch_size)
    if show_progress:
        indices = tqdm(indices, desc="Embedding documents")

    with torch.inference_mode():
        for offset in indices:
            group = documents[offset : offset + batch_size]
            tokens, mask, owners = tensorize_docs(group, tokenizer, device)
            vectors.append(model(tokens, mask, owners, len(group)).cpu().numpy())

    return np.concatenate(vectors, axis=0) if vectors else np.zeros((0, 0), np.float32)

### 4.6 Lexical candidate retrieval

In [14]:
# ------------------------------- Sparse candidate retrieval --------------- #


class LexicalRetriever:
    """Split-local word + character TF-IDF; never indexes outside-split cases."""

    def __init__(self, candidates: dict[str, dict[str, Any]], max_chars: int):
        self.ids = list(candidates)
        self.records = candidates
        self.max_chars = max_chars
        corpus = [document_text(record)[:max_chars] for record in candidates.values()]
        corpus = [text if text.strip() else " " for text in corpus]

        self.word = TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            token_pattern=r"(?u)\b\w+\b",
            sublinear_tf=True,
            min_df=1,
            max_features=45000,
            dtype=np.float32,
        )
        self.char = TfidfVectorizer(
            lowercase=True,
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=1,
            max_features=70000,
            dtype=np.float32,
        )
        self.word_corpus = self.word.fit_transform(corpus)
        self.char_corpus = self.char.fit_transform(corpus)

    def score(self, query: dict[str, Any]) -> np.ndarray:
        query_text = document_text(query)[: self.max_chars]
        word_query = self.word.transform([query_text])
        char_query = self.char.transform([query_text])
        word_scores = (word_query @ self.word_corpus.T).toarray().ravel()
        char_scores = (char_query @ self.char_corpus.T).toarray().ravel()
        return 0.60 * word_scores + 0.40 * char_scores

    def shortlist(self, query: dict[str, Any], n: int):
        scores = self.score(query)
        n = min(n, len(scores))
        indices = np.argpartition(-scores, n - 1)[:n]
        indices = indices[np.lexsort((indices, -scores[indices]))]
        return [self.ids[int(index)] for index in indices], scores[indices]

### 4.7 Micro-F1@7 scoring and development-selected reranking

In [15]:
# ------------------------------- Scoring ------------------------------------ #


def micro_at_7(predictions: dict[str, list[str]], queries: dict[str, dict[str, Any]]) -> dict[str, float]:
    if set(predictions) != set(queries):
        raise ValueError("Evaluation predictions must cover exactly the query IDs")

    true_positives = 0
    gold_count = 0
    for query_id, query in queries.items():
        predicted = predictions[query_id][:7]
        gold = set(relevant_case_ids(query))
        true_positives += len(set(predicted) & gold)
        gold_count += len(gold)

    predicted_count = 7 * len(predictions)
    precision = true_positives / predicted_count if predicted_count else 0.0
    recall = true_positives / gold_count if gold_count else 0.0
    f1 = 2 * true_positives / (predicted_count + gold_count) if predicted_count + gold_count else 0.0

    return {
        "micro_f1_at_7": float(f1),
        "micro_precision_at_7": float(precision),
        "micro_recall_at_7": float(recall),
        "true_positives": int(true_positives),
        "predicted_count": int(predicted_count),
        "gold_count": int(gold_count),
    }


def normalize_query_scores(values: np.ndarray) -> np.ndarray:
    scores = np.asarray(values, dtype=np.float32)
    return (scores - scores.mean()) / max(float(scores.std()), 1e-6)


def sparse_predictions(
    query_data: dict[str, dict[str, Any]],
    lexical: LexicalRetriever,
    limit_queries: int | None = None,
) -> dict[str, list[str]]:
    ids = list(query_data)[:limit_queries]
    return {query_id: lexical.shortlist(query_data[query_id], 7)[0] for query_id in ids}


def prepare_reranking_inputs(queries, lexical, rerank_n):
    entries = {}
    candidate_set = set()
    for query_id, record in queries.items():
        shortlist_ids, lexical_scores = lexical.shortlist(record, rerank_n)
        entries[query_id] = (shortlist_ids, lexical_scores)
        candidate_set.update(shortlist_ids)
    return entries, sorted(candidate_set)


def rerank_all(
    queries: dict[str, dict[str, Any]],
    lexical: LexicalRetriever,
    model: RetrievalEncoder,
    text_encoder: DocumentTokenizer,
    config: Config,
    device,
) -> dict[str, tuple[list[str], np.ndarray, np.ndarray]]:
    entries, candidate_ids = prepare_reranking_inputs(queries, lexical, config.rerank_top_n)
    id_to_index = {case_id: index for index, case_id in enumerate(candidate_ids)}
    case_vectors = embed_documents(
        model,
        [lexical.records[case_id] for case_id in candidate_ids],
        text_encoder,
        device,
        config.encode_batch_size,
    )
    query_vectors = embed_documents(
        model,
        list(queries.values()),
        text_encoder,
        device,
        config.encode_batch_size,
    )

    reranked = {}
    for index, query_id in enumerate(queries):
        shortlist_ids, lexical_scores = entries[query_id]
        indices = [id_to_index[case_id] for case_id in shortlist_ids]
        semantic_scores = case_vectors[indices] @ query_vectors[index]
        reranked[query_id] = (
            shortlist_ids,
            normalize_query_scores(lexical_scores),
            normalize_query_scores(semantic_scores),
        )

    return reranked


def predictions_from_scores(score_dict, semantic_weight: float, top_k: int = 7):
    output = {}
    for query_id, (ids, lexical_scores, semantic_scores) in score_dict.items():
        combined = (1 - semantic_weight) * lexical_scores + semantic_weight * semantic_scores
        order = np.lexsort((np.arange(len(ids)), -combined))
        output[query_id] = [ids[int(index)] for index in order[:top_k]]
    return output


def pick_best_weight(score_dict, dev_queries):
    """Select sparse/neural fusion ONLY on labelled development examples."""
    weights = (0.0, 0.15, 0.30, 0.45, 0.60, 0.75, 0.90, 1.0)
    candidates = []

    for weight in weights:
        predictions = predictions_from_scores(score_dict, weight)
        scores = micro_at_7(predictions, dev_queries)
        candidates.append((weight, scores))

    # Micro-F1 is the primary metric. Prefer fewer neural operations on exact ties.
    best_weight, best_scores = max(
        candidates,
        key=lambda item: (item[1]["micro_f1_at_7"], -item[0]),
    )
    return best_weight, best_scores, candidates


def validate_submission(predictions, test_queries, test_candidates, *, top_k=7):
    assert set(predictions) == set(test_queries), "Missing or extra test query IDs"
    allowed = set(test_candidates)

    for query_id, retrieved in predictions.items():
        assert isinstance(retrieved, list), f"Invalid list for query {query_id}"
        assert len(retrieved) == top_k, f"Expected exactly seven IDs for {query_id}"
        assert len(set(retrieved)) == top_k, f"Duplicate candidates for {query_id}"
        assert all(isinstance(cid, str) for cid in retrieved), "Candidate IDs must be strings"
        assert set(retrieved).issubset(allowed), f"Out-of-pool ID for query {query_id}"


def write_prediction_zip(predictions, config: Config, queries, candidates) -> tuple[Path, Path]:
    validate_submission(predictions, queries, candidates, top_k=config.top_k)
    output_dir = Path(config.work_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    json_path = output_dir / "prediction.json"
    zip_path = output_dir / f"phase7_{config.experiment}_submission.zip"

    with open(json_path, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle, indent=2, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(json_path, arcname="prediction.json")

    with zipfile.ZipFile(zip_path) as archive:
        assert archive.namelist() == ["prediction.json"]

    return json_path, zip_path

### 4.8 Positive/hard-negative construction and ranking loss

In [16]:
# -------------------------------- Pair construction ------------------------- #


class RetrievalTrainingPairs:
    """Positive and hard/random negative pairs, all from TRAIN candidates only."""

    def __init__(self, queries, candidates, lexical, seed=42, max_queries=500):
        self.queries = queries
        self.candidates = candidates
        self.lexical = lexical
        self.all_ids = list(candidates)
        self.rng = random.Random(seed)

        eligible = []
        for query_id, record in queries.items():
            relevant = [cid for cid in relevant_case_ids(record) if cid in candidates]
            if relevant:
                eligible.append((query_id, relevant))

        self.rng.shuffle(eligible)
        self.eligible = eligible[:max_queries] if max_queries > 0 else eligible
        if not self.eligible:
            raise ValueError("No train query has a labelled positive in the TRAIN candidate pool")

        self.hard_negative_ids = {}
        for query_id, positives in tqdm(self.eligible, desc="Mining TRAIN negatives"):
            retrieved, _ = lexical.shortlist(queries[query_id], 35)
            positives = set(positives)
            self.hard_negative_ids[query_id] = [cid for cid in retrieved if cid not in positives]

    def epoch_examples(self):
        items = list(self.eligible)
        self.rng.shuffle(items)
        for index, (query_id, positive_ids) in enumerate(items):
            positive_id = self.rng.choice(positive_ids)
            gold = set(relevant_case_ids(self.queries[query_id]))
            hard = self.hard_negative_ids[query_id]

            if hard and index % 4 != 0:
                negative_id = self.rng.choice(hard[:15])
            else:
                negative_id = self.rng.choice(self.all_ids)
                for _ in range(100):
                    if negative_id not in gold:
                        break
                    negative_id = self.rng.choice(self.all_ids)
                if negative_id in gold:
                    continue

            yield (query_id, positive_id, negative_id)


def contrastive_logits(queries, positives, negatives, temperature: float):
    candidates = torch.cat((positives, negatives), dim=0)
    return (queries @ candidates.T) / temperature


def batch_documents(examples, train_queries, train_candidates):
    query_docs = [train_queries[query_id] for query_id, _, _ in examples]
    positive_docs = [train_candidates[positive_id] for _, positive_id, _ in examples]
    negative_docs = [train_candidates[negative_id] for _, _, negative_id in examples]
    return query_docs + positive_docs + negative_docs


def training_step(model, teacher, examples, train_queries, train_candidates, doc_tokenizer, cfg, device):
    batch_size = len(examples)
    docs = batch_documents(examples, train_queries, train_candidates)
    token_ids, attention_mask, owners = tensorize_docs(docs, doc_tokenizer, device)

    vectors = model(token_ids, attention_mask, owners, len(docs))
    q, pos, neg = vectors.split(batch_size, dim=0)
    scores = contrastive_logits(q, pos, neg, cfg.temperature)
    labels = torch.arange(batch_size, device=device)

    # Avoid false negatives across both other rows' positives and negatives.
    candidate_ids = [pos_id for _, pos_id, _ in examples] + [
        neg_id for _, _, neg_id in examples
    ]
    for row, (query_id, _, _) in enumerate(examples):
        relevant = set(relevant_case_ids(train_queries[query_id]))
        for column, candidate_id in enumerate(candidate_ids):
            if column != row and candidate_id in relevant:
                scores[row, column] = -1e4

    objective = F.cross_entropy(scores, labels)

    if teacher is not None:
        with torch.no_grad():
            teacher_vectors = teacher(token_ids, attention_mask, owners, len(docs))
            tq, tp, tn = teacher_vectors.split(batch_size, dim=0)
            teacher_logits = contrastive_logits(tq, tp, tn, cfg.temperature)
            for row, (query_id, _, _) in enumerate(examples):
                relevant = set(relevant_case_ids(train_queries[query_id]))
                for column, candidate_id in enumerate(candidate_ids):
                    if column != row and candidate_id in relevant:
                        teacher_logits[row, column] = -1e4


        t = cfg.kd_temperature
        distillation = F.kl_div(
            F.log_softmax(scores / t, dim=-1),
            F.softmax(teacher_logits / t, dim=-1),
            reduction="batchmean",
        ) * t * t
        objective = objective + cfg.kd_weight * distillation

    return objective


def train_epoch(model, teacher, pairs, queries, candidates, doc_tokenizer, optimizer, scaler, cfg, device):
    model.train()
    if teacher is not None:
        teacher.eval()

    total = 0.0
    n_batches = 0
    optimizer.zero_grad(set_to_none=True)
    group = []

    def flush_group(batch, step_index):
        amp_enabled = cfg.use_amp and device.type == "cuda"
        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            loss = training_step(
                model, teacher, batch, queries, candidates, doc_tokenizer, cfg, device
            )
            scaled_loss = loss / cfg.accumulate_steps

        if scaler is None:
            scaled_loss.backward()
        else:
            scaler.scale(scaled_loss).backward()

        if (step_index + 1) % cfg.accumulate_steps == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        return float(loss.detach().cpu())

    iterator = pairs.epoch_examples()
    for example in iterator:
        group.append(example)
        if len(group) < cfg.batch_size:
            continue

        total += flush_group(group, n_batches)
        n_batches += 1
        group = []
        if n_batches >= cfg.max_steps_per_epoch:
            break

    if group and n_batches < cfg.max_steps_per_epoch:
        total += flush_group(group, n_batches)
        n_batches += 1

    # Final incomplete gradient accumulation group is still stepped.
    if n_batches > 0 and n_batches % cfg.accumulate_steps:
        if scaler is not None:
            scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        if scaler is not None:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    return total / max(n_batches, 1)


def parameter_counts(model: nn.Module) -> dict[str, int]:
    return {
        "total": sum(p.numel() for p in model.parameters()),
        "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad),
    }


def save_best_model(path, model, config, metrics, weight, epoch):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": asdict(config),
            "dev_metrics": metrics,
            "dev_fusion_weight": weight,
            "epoch": epoch,
        },
        path,
    )


def choose_optimizer(model: RetrievalEncoder, config: Config):
    base = []
    head = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("projection.") or ".a.weight" in name or ".b.weight" in name or ".scales" in name:
            head.append(parameter)
        else:
            base.append(parameter)

    groups = []
    if base:
        groups.append({"params": base, "lr": config.backbone_learning_rate})
    if head:
        groups.append({"params": head, "lr": config.learning_rate})

    return torch.optim.AdamW(groups, weight_decay=config.weight_decay)


def find_asset(config: Config, name: str) -> Path:
    root = Path(config.asset_dir)
    if root.is_file() and root.name == name:
        return root
    if root.is_dir():
        found = sorted(root.rglob(name))
        if len(found) == 1:
            return found[0]
        if len(found) > 1:
            raise FileNotFoundError(
                f"Multiple files called {name!r} found. Set ASSET_DIR to the exact "
                f"Phase-1 dataset folder, not {root}"
            )
    raise FileNotFoundError(
        f"Cannot find {name!r} under {root}. Attach the original Phase-1 Model 2 "
        "checkpoint and its custom_bpe_scratch.json tokenizer as Kaggle inputs."
    )


def fit_experiment(config: Config, data, model, doc_tokenizer, checkpoint_path, device):
    output_root = Path(config.work_dir)
    output_root.mkdir(parents=True, exist_ok=True)
    train_lexical = LexicalRetriever(data["train_candidates"], config.lexical_chars)
    dev_lexical = LexicalRetriever(data["dev_candidates"], config.lexical_chars)

    dev_ids = list(data["dev_queries"])
    random.Random(config.seed).shuffle(dev_ids)
    if config.dev_queries_per_epoch > 0:
        dev_ids = dev_ids[: config.dev_queries_per_epoch]
    dev_subset = {qid: data["dev_queries"][qid] for qid in dev_ids}

    pairs = RetrievalTrainingPairs(
        data["train_queries"],
        data["train_candidates"],
        train_lexical,
        seed=config.seed,
        max_queries=config.max_train_queries,
    )

    teacher = None
    if config.experiment == "kd":
        if not config.teacher_checkpoint or not Path(config.teacher_checkpoint).is_file():
            raise FileNotFoundError(
                "KD requires a trained Phase-7 FULL retrieval teacher checkpoint. "
                "Set TEACHER_CHECKPOINT to the full experiment's best_model.pt. "
                "Do not substitute an untrained Phase-1 backbone as teacher."
            )
        teacher = make_model(config, checkpoint_path, method="full").to(device)
        checkpoint = torch.load(config.teacher_checkpoint, map_location="cpu", weights_only=False)
        teacher.load_state_dict(checkpoint["model_state_dict"], strict=True)
        teacher.requires_grad_(False)
        teacher.eval()
        if config.student_layers >= config.n_layers:
            raise ValueError("KD student must use fewer layers than its Phase-1 teacher")

    optimizer = choose_optimizer(model, config)
    amp_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" and config.use_amp else None
    best_metric = -1.0
    best_path = output_root / "best_model.pt"
    history = []
    best_epoch = 0

    print("Parameters:", parameter_counts(model))
    print("Epoch selection uses DEVELOPMENT Micro-F1@7 only.")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(1, config.epochs + 1):
        started = time.monotonic()
        loss = train_epoch(
            model,
            teacher,
            pairs,
            data["train_queries"],
            data["train_candidates"],
            doc_tokenizer,
            optimizer,
            amp_scaler,
            config,
            device,
        )
        score_dict = rerank_all(
            dev_subset, dev_lexical, model, doc_tokenizer, config, device
        )
        fusion_weight, metrics, all_weights = pick_best_weight(score_dict, dev_subset)
        elapsed = time.monotonic() - started

        result = {
            "epoch": epoch,
            "train_loss": loss,
            "dev_micro_f1_at_7": metrics["micro_f1_at_7"],
            "dev_precision_at_7": metrics["micro_precision_at_7"],
            "dev_recall_at_7": metrics["micro_recall_at_7"],
            "selected_semantic_weight": fusion_weight,
            "epoch_seconds": elapsed,
            "peak_cuda_memory_bytes": (
                int(torch.cuda.max_memory_allocated(device))
                if device.type == "cuda"
                else 0
            ),
        }
        history.append(result)
        print(json.dumps(result, indent=2))

        if metrics["micro_f1_at_7"] > best_metric + 1e-12:
            best_metric = metrics["micro_f1_at_7"]
            best_epoch = epoch
            save_best_model(best_path, model, config, metrics, fusion_weight, epoch)
            with open(output_root / "dev_fusion_trials.json", "w") as handle:
                json.dump([{"weight": w, **m} for w, m in all_weights], handle, indent=2)

        if epoch - best_epoch >= config.patience:
            print("Early stopping: no improvement in dev Micro-F1@7.")
            break

    with open(output_root / "training_history.json", "w") as handle:
        json.dump(history, handle, indent=2)

    snapshot = torch.load(best_path, map_location="cpu", weights_only=False)
    model.load_state_dict(snapshot["model_state_dict"], strict=True)
    return model, float(snapshot["dev_fusion_weight"]), history


def final_evaluation_and_submission(config, data, model, doc_tokenizer, device, fusion_weight):
    output_dir = Path(config.work_dir)
    dev_lexical = LexicalRetriever(data["dev_candidates"], config.lexical_chars)
    dev_queries = data["dev_queries"]
    dev_raw = rerank_all(dev_queries, dev_lexical, model, doc_tokenizer, config, device)
    dev_predictions = predictions_from_scores(dev_raw, fusion_weight)
    dev_metrics = micro_at_7(dev_predictions, dev_queries)

    baseline_predictions = sparse_predictions(dev_queries, dev_lexical)
    baseline_metrics = micro_at_7(baseline_predictions, dev_queries)

    # Diagnostic: how many gold cases can any reranker possibly find if it is
    # restricted to the current lexical top-N shortlist?
    oracle_hits = 0
    oracle_gold = 0
    error_rows = []
    for query_id, record in dev_queries.items():
        gold = set(relevant_case_ids(record))
        shortlisted = set(dev_raw[query_id][0])
        reranked_hits = len(set(dev_predictions[query_id]) & gold)
        baseline_hits = len(set(baseline_predictions[query_id]) & gold)
        oracle_hits += min(config.top_k, len(gold & shortlisted))
        oracle_gold += len(gold)
        error_rows.append({
            "query_id": query_id,
            "query_sentence_count": len(document_sentences(record)),
            "gold_count": len(gold),
            "baseline_top7_hits": baseline_hits,
            "neural_top7_hits": reranked_hits,
            "retrievable_gold_in_shortlist": len(gold & shortlisted),
        })

    import csv

    with open(output_dir / "dev_error_analysis.csv", "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(error_rows[0]))
        writer.writeheader()
        writer.writerows(error_rows)

    oracle_denom = config.top_k * len(dev_queries) + oracle_gold
    oracle_f1 = 2 * oracle_hits / oracle_denom if oracle_denom else 0.0

    with open(output_dir / "dev_predictions.json", "w") as handle:
        json.dump(dev_predictions, handle, indent=2)
    with open(output_dir / "dev_candidate_ids.json", "w") as handle:
        json.dump(list(data["dev_candidates"]), handle, indent=2)
    with open(output_dir / "dev_truth.json", "w") as handle:
        json.dump({qid: relevant_case_ids(row) for qid, row in dev_queries.items()}, handle, indent=2)

    report = {
        "method": config.experiment,
        "primary_metric": "Micro-F1@7",
        "lexical_baseline": baseline_metrics,
        "adapted_model": dev_metrics,
        "semantic_fusion_weight": fusion_weight,
        "parameter_counts": parameter_counts(model),
        "full_dev_query_count": len(dev_queries),
        "lexical_top_n_oracle_f1_at_7": float(oracle_f1),
        "training_seconds": float(sum(row["epoch_seconds"] for row in json.loads(
            (output_dir / "training_history.json").read_text()
        ))),
        "peak_cuda_memory_bytes": int(max((row["peak_cuda_memory_bytes"] for row in json.loads(
            (output_dir / "training_history.json").read_text()
        )), default=0)),
        "note": "These are development scores, not hidden Codabench test scores.",
    }
    with open(output_dir / "metrics.json", "w") as handle:
        json.dump(report, handle, indent=2)
    print("Full development report:", json.dumps(report, indent=2))

    if not config.make_test_predictions:
        return report, None

    test_lexical = LexicalRetriever(data["test_candidates"], config.lexical_chars)
    test_raw = rerank_all(
        data["test_queries"], test_lexical, model, doc_tokenizer, config, device
    )
    predictions = predictions_from_scores(test_raw, fusion_weight)
    json_path, zip_path = write_prediction_zip(
        predictions,
        config,
        data["test_queries"],
        data["test_candidates"],
    )
    print("Created:", json_path)
    print("Codabench ZIP:", zip_path)
    return report, zip_path


def run(config: Config):
    config.__post_init__()
    seed_everything(config.seed)
    output_dir = Path(config.work_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with open(output_dir / "config.json", "w") as handle:
        json.dump(asdict(config), handle, indent=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device, "| method:", config.experiment)
    print("Loading official Phase-7 dataset...")
    data = load_official_data(config)

    phase1_checkpoint = find_asset(config, config.checkpoint_name)
    tokenizer_file = find_asset(config, config.tokenizer_name)
    tokenizer = CustomByteLevelBPE(config.vocab_size)
    tokenizer.load(tokenizer_file)
    print("Strict Phase-1 checkpoint:", phase1_checkpoint)
    print("Matching Phase-1 tokenizer:", tokenizer_file)

    method = "kd_student" if config.experiment == "kd" else config.experiment
    model = make_model(config, phase1_checkpoint, method=method).to(device)
    document_encoder = DocumentTokenizer(tokenizer, config)

    model, fusion_weight, history = fit_experiment(
        config, data, model, document_encoder, phase1_checkpoint, device
    )
    report, archive_path = final_evaluation_and_submission(
        config, data, model, document_encoder, device, fusion_weight
    )
    return {"metrics": report, "history": history, "submission": archive_path}

## 5. Run training, select the best development checkpoint, and export

Running this cell downloads the six official Phase-7 files, **strictly loads your Phase-1 weights**, builds TRAIN hard negatives, fine-tunes on legal query–case relevance, chooses the strongest checkpoint by development Micro-F1@7, evaluates on the full development split, and exports the test `prediction.json` inside a root-level ZIP.

The script writes `best_model.pt`, `training_history.json`, `dev_fusion_trials.json`, `metrics.json`, `dev_predictions.json`, `dev_truth.json` and `dev_candidate_ids.json` under `OUTPUT_DIR`. The `metrics.json` figures are **development** metrics, not hidden-test leaderboard values.


In [17]:
CONFIG = Config(
    experiment=EXPERIMENT,
    asset_dir=ASSET_DIR,
    data_dir=DATA_DIR,
    work_dir=OUTPUT_DIR,
    teacher_checkpoint=TEACHER_CHECKPOINT,
    vocab_size=MODEL_VOCAB_SIZE,
    hidden_dim=MODEL_HIDDEN_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=MODEL_HEADS,
    n_kv_heads=MODEL_KV_HEADS,
    epochs=TRAIN_EPOCHS,
    max_train_queries=MAX_TRAIN_QUERIES,
    max_steps_per_epoch=STEPS_PER_EPOCH,
    batch_size=TRAIN_BATCH_SIZE,
    accumulate_steps=ACCUMULATE_STEPS,
    rerank_top_n=RERANK_TOP_N,
    dev_queries_per_epoch=DEV_QUERIES_PER_EPOCH,
    chunks_per_doc=CHUNKS_PER_CASE,
    tokens_per_chunk=TOKENS_PER_CHUNK,
)

RUN_RESULT = run(CONFIG)

print("\nExperiment completed:", EXPERIMENT)
print("Validation Micro-F1@7:", RUN_RESULT["metrics"]["adapted_model"]["micro_f1_at_7"])
print("Submission ZIP:", RUN_RESULT["submission"])

Device: cuda | method: full
Loading official Phase-7 dataset...


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (4 outputs left out)

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Train:   827 queries;   4320 candidates
Dev  :   118 queries;   1023 candidates
Test :   237 queries;   1727 candidates
Strict Phase-1 checkpoint: /kaggle/input/datasets/kalyanbabuk/phase1/pytorch_model.bin
Matching Phase-1 tokenizer: /kaggle/input/datasets/kalyanbabuk/phase1/custom_bpe_scratch.json


Mining TRAIN negatives:   0%|          | 0/500 [00:00<?, ?it/s]

Parameters: {'total': 59797504, 'trainable': 59797504}
Epoch selection uses DEVELOPMENT Micro-F1@7 only.
{
  "epoch": 1,
  "train_loss": 1.3862267127403847,
  "dev_micro_f1_at_7": 0.27184466019417475,
  "dev_precision_at_7": 0.3333333333333333,
  "dev_recall_at_7": 0.22950819672131148,
  "selected_semantic_weight": 0.0,
  "epoch_seconds": 59.558543052999994,
  "peak_cuda_memory_bytes": 1226552832
}
{
  "epoch": 2,
  "train_loss": 1.3842623197115385,
  "dev_micro_f1_at_7": 0.27184466019417475,
  "dev_precision_at_7": 0.3333333333333333,
  "dev_recall_at_7": 0.22950819672131148,
  "selected_semantic_weight": 0.0,
  "epoch_seconds": 41.79716828400001,
  "peak_cuda_memory_bytes": 1226552832
}
{
  "epoch": 3,
  "train_loss": 1.3865985576923077,
  "dev_micro_f1_at_7": 0.27184466019417475,
  "dev_precision_at_7": 0.3333333333333333,
  "dev_recall_at_7": 0.22950819672131148,
  "selected_semantic_weight": 0.0,
  "epoch_seconds": 39.992580817999965,
  "peak_cuda_memory_bytes": 1226552832
}
Early

## 6. Review the metrics and perform a final submission check

**Micro-F1@7** = `2 × correct_retrievals / (7 × number_of_queries + number_of_gold_relevant_cases)`.

Compare full development Micro-F1@7 against the lexical baseline. A selected semantic weight of `0.0` means lexical retrieval was stronger than neural reranking on the measured development subset, not that the neural model was unavailable. Inspect `dev_fusion_trials.json`, compare false negatives, and decide whether more training or a different representation is justified.

The generated ZIP contains `prediction.json` at its root. Submission validation prohibits duplicate IDs, missing queries and cross-split candidates.

**KD caveat:** A student score must be compared with the corresponding fully trained Phase-7 teacher before claiming compression success.


In [18]:
from pathlib import Path
import json
import zipfile

output = Path(OUTPUT_DIR)
with open(output / "metrics.json", encoding="utf-8") as handle:
    final_metrics = json.load(handle)

with open(output / "prediction.json", encoding="utf-8") as handle:
    test_predictions = json.load(handle)

with zipfile.ZipFile(RUN_RESULT["submission"]) as archive:
    assert archive.namelist() == ["prediction.json"]

assert all(len(ids) == 7 and len(set(ids)) == 7 for ids in test_predictions.values())
print(json.dumps(final_metrics, indent=2))
print(f"Validated {len(test_predictions)} test query predictions.")

{
  "method": "full",
  "primary_metric": "Micro-F1@7",
  "lexical_baseline": {
    "micro_f1_at_7": 0.30127848804891605,
    "micro_precision_at_7": 0.3280871670702179,
    "micro_recall_at_7": 0.27852004110996914,
    "true_positives": 271,
    "predicted_count": 826,
    "gold_count": 973
  },
  "adapted_model": {
    "micro_f1_at_7": 0.30127848804891605,
    "micro_precision_at_7": 0.3280871670702179,
    "micro_recall_at_7": 0.27852004110996914,
    "true_positives": 271,
    "predicted_count": 826,
    "gold_count": 973
  },
  "semantic_fusion_weight": 0.0,
  "parameter_counts": {
    "total": 59797504,
    "trainable": 59797504
  },
  "full_dev_query_count": 118,
  "lexical_top_n_oracle_f1_at_7": 0.5180655919955531,
  "training_seconds": 141.34829215499997,
  "peak_cuda_memory_bytes": 1226552832,
  "note": "These are development scores, not hidden Codabench test scores."
}
Validated 237 test query predictions.
